# Bull Re-entry Joint Target Contract v1

Purpose: verify that the fold-local joint target contract can call the bull re-entry participation target gate after sparse-release/drawdown-control target rows have been collapsed. This is a target-generation handoff toward retraining; it is not an actor promotion result.

Governance:
- Canonical baseline: `soxl_backtest_policy_reimplementation_v1` as qlib reference baseline, not a deployable actor.
- Direct comparison baseline: current strict walk-forward gate rows in `runtime/model_selection/card/model_retention_walkforward_gate_v1/summary.json`.
- Role: diagnose/prepare next training branch.
- Target window: fold-local train rows for `wf_2020`, `wf_2022`, `wf_2023_2024`; this notebook uses a deterministic contract fixture that exercises train/eval leakage boundaries.
- Changed layer: target generation.
- Acceptance: bull re-entry rows lift train-only BUY/release states, skip eval overlap and drawdown-control/downtrend conflicts, and keep the target-scope audit train-ready.
- Failure mode: if the gate is only a standalone helper and not reachable from `build_fold_local_joint_target_contract`, the next BC/PPO retraining branch cannot consume it safely.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "AGENTS.md").exists() or not (ROOT / "SPEC.md").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("repo root not found")
    ROOT = ROOT.parent
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.state_objective_training_targets import (
    BullReentryParticipationTargetConfig,
    audit_state_objective_target_scope,
    build_fold_local_joint_target_contract,
    summarize_state_objective_target_scope_audit,
)

CARD = ROOT / "runtime" / "model_selection" / "card" / "bull_reentry_joint_target_contract_v1"
CARD.mkdir(parents=True, exist_ok=True)

In [2]:
primary = pd.DataFrame({
    "datetime": pd.to_datetime(["2020-03-24", "2022-03-24"]),
    "fold": ["wf_2020_train", "wf_2020_train"],
    "target_scope": ["fold_local", "fold_local"],
    "action": ["BUY", "BUY"],
    "teacher_position": [0.20, 0.20],
    "rebound_target_position": [0.20, 0.20],
    "risk_target_position": [0.20, 0.20],
    "risk_target_position_original": [0.20, 0.20],
    "is_rebound_cluster": [False, False],
    "is_downtrend_state": [False, False],
    "release_gate_state": [False, False],
    "drawdown_control_state": [False, False],
})
auxiliary = pd.DataFrame({
    "datetime": pd.to_datetime(["2020-04-01", "2020-04-03"]),
    "fold": ["wf_2020_train", "wf_2020_train"],
    "target_scope": ["fold_local", "fold_local"],
    "action": ["HOLD", "BUY"],
    "teacher_position": [0.35, 0.50],
    "rebound_target_position": [0.35, 0.50],
    "risk_target_position": [0.35, 0.50],
    "risk_target_position_original": [0.35, 0.50],
    "is_rebound_cluster": [False, True],
    "is_downtrend_state": [False, True],
    "release_gate_state": [True, True],
    "drawdown_control_state": [False, True],
})
folds = [{
    "name": "wf_2020_train",
    "train_start": "2020-01-01",
    "train_end": "2020-12-31",
    "eval_start": "2022-01-01",
    "eval_end": "2022-12-31",
}]

config = BullReentryParticipationTargetConfig(
    enabled=True,
    min_target_position=0.70,
    max_target_position=0.85,
    objective_weight_bonus=5.0,
)
joint = build_fold_local_joint_target_contract(
    primary,
    auxiliary,
    folds=folds,
    bull_reentry_config=config,
)
joint

,datetime,candidate,window,fold,target_scope,train_start,train_end,eval_start,eval_end,teacher_position,...,source_components,source_component_count,source_row_count,rebound_participation_weight,bull_reentry_signal_ready,bull_reentry_train_window_ready,bull_reentry_target_state,bull_reentry_target_position,bull_reentry_target_lift,bull_reentry_target_reason
0,2020-03-24,joint_target_contract,wf_2020_train,wf_2020_train,fold_local,2020-01-01,2020-12-31,2022-01-01,2022-12-31,0.20,...,primary,1,1,5.0,True,True,True,0.7,0.50,fold_local_bull_reentry_participation
1,2020-04-01,joint_target_contract,wf_2020_train,wf_2020_train,fold_local,2020-01-01,2020-12-31,2022-01-01,2022-12-31,0.35,...,auxiliary,1,1,5.0,True,True,True,0.7,0.35,fold_local_bull_reentry_participation
2,2020-04-03,joint_target_contract,wf_2020_train,wf_2020_train,fold_local,2020-01-01,2020-12-31,2022-01-01,2022-12-31,0.50,...,auxiliary,1,1,0.0,True,True,False,0.5,0.00,


In [3]:
audit = audit_state_objective_target_scope(joint, folds=folds)
scope_summary = summarize_state_objective_target_scope_audit(audit)
active = joint[joint["bull_reentry_target_state"].fillna(False).astype(bool)]
risk_conflicts = joint[
    joint["bull_reentry_target_state"].fillna(False).astype(bool)
    & (joint["is_downtrend_state"].fillna(False).astype(bool) | joint["drawdown_control_state"].fillna(False).astype(bool))
]
summary = {
    "experiment": "bull_reentry_joint_target_contract_v1",
    "decision": "joint_target_contract_bull_reentry_gate_ready_for_training_handoff",
    "training_allowed_next": bool(scope_summary["training_allowed"] and len(active) > 0 and risk_conflicts.empty),
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "rows": int(len(joint)),
    "bull_reentry_target_rows": int(len(active)),
    "eval_overlap_rows": int(scope_summary["eval_overlap_rows"]),
    "risk_conflict_target_rows": int(len(risk_conflicts)),
    "max_target_lift": float(joint.get("bull_reentry_target_lift", pd.Series(dtype=float)).max()),
    "mean_active_target_position": float(active["bull_reentry_target_position"].mean()) if not active.empty else 0.0,
    "scope_decision": scope_summary["decision"],
    "next_mechanism": "wire_fold_local_bull_reentry_joint_targets_into_bc_warm_start_then_full_walkforward_gate",
    "required_retention_gates": {
        "required_wf_windows": ["wf_2020", "wf_2022", "wf_2023_2024"],
        "max_walkforward_drawdown_pct": 20.0,
        "reference_baseline": "soxl_backtest_policy_reimplementation_v1",
        "required_best_return_pct": 213.143462,
    },
}
(CARD / "summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
joint.to_csv(CARD / "joint_target_contract.csv", index=False)
audit.to_csv(CARD / "scope_audit.csv", index=False)
summary

{'experiment': 'bull_reentry_joint_target_contract_v1',
 'decision': 'joint_target_contract_bull_reentry_gate_ready_for_training_handoff',
 'training_allowed_next': True,
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'rows': 3,
 'bull_reentry_target_rows': 2,
 'eval_overlap_rows': 0,
 'risk_conflict_target_rows': 0,
 'max_target_lift': 0.5,
 'mean_active_target_position': 0.7,
 'scope_decision': 'state_objective_targets_training_ready',
 'next_mechanism': 'wire_fold_local_bull_reentry_joint_targets_into_bc_warm_start_then_full_walkforward_gate',
 'required_retention_gates': {'required_wf_windows': ['wf_2020',
   'wf_2022',
   'wf_2023_2024'],
  'max_walkforward_drawdown_pct': 20.0,
  'reference_baseline': 'soxl_backtest_policy_reimplementation_v1',
  'required_best_return_pct': 213.143462}}

## Conclusion

The joint target contract now exposes the bull re-entry participation gate at the training-contract boundary. The fixture proves two train-window rows are lifted to a 0.70 target, the eval-overlap row is absent, and the downtrend/drawdown-control conflict row is not lifted.

Decision: `joint_target_contract_bull_reentry_gate_ready_for_training_handoff`.

Promotion: blocked. This is target-generation evidence only. Actor retention still requires full walk-forward windows (`wf_2020`, `wf_2022`, `wf_2023_2024`), max walk-forward drawdown below 20%, and return alignment against the SOXLBacktest reference/current-best threshold.

Next action: wire these fold-local joint targets into a BC warm-start candidate, then run the strict walk-forward retention gate before keeping any actor.